In [ ]:
pip install pgmpy

In [ ]:
pip install networkx matplotlib

In [ ]:
import pgmpy.models
import pgmpy.inference
import networkx as nx
import pylab as plt
# Create a bayesian network
model = pgmpy.models.BayesianModel([('Guest', 'Monty'),
                                    ('Price', 'Monty')])
# Define conditional probability distributions (CPD)
# Probability of guest selecting door 0, 1 and 2
cpd_guest = pgmpy.factors.discrete.TabularCPD('Guest', 3, [[0.33], [0.33], [0.33]])
# Probability that the price is behind door 0, 1 and 2
cpd_price = pgmpy.factors.discrete.TabularCPD('Price', 3, [[0.33], [0.33], [0.33]])
# Probability that Monty selects a door (0, 1, 2), when we know which door the guest has selected and we know were the price is
cpd_monty = pgmpy.factors.discrete.TabularCPD('Monty', 3, [[0, 0, 0, 0, 0.5, 1, 0, 1, 0.5],
                                                           [0.5, 0, 1, 0, 0, 0, 1, 0, 0.5],
                                                           [0.5, 1, 0, 1, 0.5, 0, 0, 0, 0]],
                                              evidence=['Guest', 'Price'],
                                              evidence_card=[3, 3])
# Add CPDs to the network structure
model.add_cpds(cpd_guest, cpd_price, cpd_monty)
# Check if the model is valid, throw an exception otherwise
model.check_model()
# Print probability distributions
print('Probability distribution, P(Guest)')
print(cpd_guest)
print()
print('Probability distribution, P(Price)')
print(cpd_price)
print()
print('Joint probability distribution, P(Monty | Guest, Price)')
print(cpd_monty)
print()
# Plot the model
#nx.draw(model, with_labels=True)
#plt.savefig('C:\\DATA\\Python-data\\bayesian-networks\\monty-hall.png')
plt.close()
# Perform variable elimination for inference
# Variable elimination (VE) is a an exact inference algorithm in bayesian networks
infer = pgmpy.inference.VariableElimination(model)
# Calculate probabilites for doors including price, the guest has selected door 0 and Monty has selected door 2
posterior_probability = infer.query(['Price'], evidence={'Guest': 0, 'Monty': 2})
# Print posterior probability
print('Posterior probability, Guest(0) and Monty(2)')
print(posterior_probability)
print()

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


AttributeError: module 'pgmpy.models' has no attribute 'BayesianModel'

# Monty Hall Bayesian Network - Code Explanation

```python
import pgmpy.models
import pgmpy.inference
import networkx as nx
import pylab as plt
```
**Imports**:  
- `pgmpy.models`: Provides Bayesian Network model classes.  
- `pgmpy.inference`: Contains inference algorithms like Variable Elimination.  
- `networkx` & `pylab`: Used for visualizing the network graph (commented out in this code).

```python
# Create a bayesian network
model = pgmpy.models.BayesianModel([('Guest', 'Monty'),
                                    ('Price', 'Monty')])
```
**Model Definition**:  
Creates a Bayesian Network with two edges:  
- `Guest → Monty` (Monty's choice depends on the guest's door)  
- `Price → Monty` (Monty's choice depends on where the prize is)

```python
# Define conditional probability distributions (CPD)
# Probability of guest selecting door 0, 1 and 2
cpd_guest = pgmpy.factors.discrete.TabularCPD('Guest', 3, [[0.33], [0.33], [0.33]])
```
**Guest CPD**:  
Defines the prior probability distribution for the `Guest` variable (3 doors).  
Each door has equal probability: 0.33 (approximately 1/3).

```python
# Probability that the price is behind door 0, 1 and 2
cpd_price = pgmpy.factors.discrete.TabularCPD('Price', 3, [[0.33], [0.33], [0.33]])
```
**Price CPD**:  
Prior probability distribution for the `Price` variable (3 doors).  
Each door has equal probability: 0.33 (prize is equally likely behind any door).

```python
# Probability that Monty selects a door (0, 1, 2), when we know which door the guest has selected and we know were the price is
cpd_monty = pgmpy.factors.discrete.TabularCPD('Monty', 3, [[0, 0, 0, 0, 0.5, 1, 0, 1, 0.5],
                                                           [0.5, 0, 1, 0, 0, 0, 1, 0, 0.5],
                                                           [0.5, 1, 0, 1, 0.5, 0, 0, 0, 0]],
                                              evidence=['Guest', 'Price'],
                                              evidence_card=[3, 3])
```
**Monty CPD**:  
Conditional probability table for Monty's door selection given Guest's choice and Prize location.  
- **3 rows**: Monty's possible choices (doors 0, 1, 2)  
- **9 columns**: All combinations of (Guest, Price) where each ranges over 3 values  
  - Column order: (Guest=0,Price=0), (0,1), (0,2), (1,0), (1,1), (1,2), (2,0), (2,1), (2,2)  
- **Values**: Probability Monty opens each door following the game rules (never reveal prize, never open guest's door, random if both are same)

```python
# Add CPDs to the network structure
model.add_cpds(cpd_guest, cpd_price, cpd_monty)
```
**Add CPDs**:  
Attaches the defined probability distributions to the Bayesian Network model.

```python
# Check if the model is valid, throw an exception otherwise
model.check_model()
```
**Model Validation**:  
Verifies that:  
- All CPDs are properly defined  
- Sum of probabilities in each CPD sums to 1  
- Network structure is acyclic (DAG)

```python
# Print probability distributions
print('Probability distribution, P(Guest)')
print(cpd_guest)
print()
print('Probability distribution, P(Price)')
print(cpd_price)
print()
print('Joint probability distribution, P(Monty | Guest, Price)')
print(cpd_monty)
print()
```
**Print CPDs**:  
Displays all three probability distributions to the console for inspection.

```python
# Plot the model
#nx.draw(model, with_labels=True)
plt.close()
```
**Visualization** (commented out):  
- `nx.draw()`: Would create a visual graph of the network  
- `plt.savefig()`: Would save the plot to a file  
- `plt.close()`: Closes any open figure windows

```python
# Perform variable elimination for inference
# Variable elimination (VE) is a an exact inference algorithm in bayesian networks
infer = pgmpy.inference.VariableElimination(model)
```
**Inference Setup**:  
Creates a Variable Elimination inference engine for the Bayesian Network.  
VE is an exact inference algorithm that computes marginal probabilities efficiently.

```python
# Calculate probabilites for doors including price, the guest has selected door 0 and Monty has selected door 2
posterior_probability = infer.query(['Price'], evidence={'Guest': 0, 'Monty': 2})
```
**Query Execution**:  
Computes the posterior probability distribution for `Price` given:  
- `Guest` chose door 0 (evidence)  
- `Monty` opened door 2 (evidence)  
This answers: "Given the guest picked door 0 and Monty opened door 2, what's the probability the prize is behind each door?"

```python
# Print posterior probability
print('Posterior probability, Guest(0) and Monty(2)')
print(posterior_probability)
print()
```
**Output Results**:  
Prints the computed posterior probabilities.  
Expected result: P(Price=1) ≈ 0.667, P(Price=0) ≈ 0.333 (illustrates the Monty Hall paradox - switching doors doubles the win probability).

<br>
<br>
<br>
<br>

This code proves why I should always switch doors on the Monty Hall game show. The program sets up the game with three doors, one has a prize, two have goats. I pick a door, then the host, who knows where the prize is, opens another door that has a goat. The code then asks: if I picked door 0 and the host opened door 2, what are my chances? The answer: if I stay with door 0, I win only 1 out of 3 times. But if I switch to the other unopened door, I win 2 out of 3 times. Why? Because when I first picked, I had a 1 in 3 chance of being right, meaning there was a 2 in 3 chance the prize was behind one of the other doors. When the host opens a goat door, that 2 in 3 chance doesn't disappeae, it just moves to the one door he didn't open. So switching turns the odds in my favor, and the code shows this with cold, hard math.